## Initial setup

Before running the notebook:
- update the .env file with your team's `OPENAI_API_KEY`
- Install uv `pip install uv`
- run `uv sync`
- run `uv run phoenix serve` (If Arize Phoenix is enabled) - takes a few minutes
- Select Kernal for this notebook

## Raven Starter Bot

This notebook contains a **Raven Starter Bot** with:

1. **Message parsing** ( `parse_message`, `parse_outgoing_message`)
2. **Role-specific random responders** (Villager, Raven, Detective, Doctor)
4. **A single async loop** `connect_parse_respond_forever()`

You can:
- Run the bot and let it play matches.
- Inspect each game later by `gameId` using the log utilities.
- Stop the bot




## Imports and basic configuration

In [1]:
import asyncio
import json
import os
import random
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any
from collections import defaultdict
import itertools
from datetime import datetime
from pathlib import Path

import websockets

from openai import AsyncOpenAI
from pydantic import BaseModel
from uuid import uuid4
from dotenv import load_dotenv


load_dotenv()

# --- WebSocket configuration ---
WS_URL = os.getenv("WS_URL", "ws://localhost:2025")
CONNECT_TIMEOUT = 10  # seconds to wait when opening
RECV_TIMEOUT = 0.1
KEEP_ALIVE = True
ENABLE_PHOENIX = True


log_file_paths = [
    {
        "game_id": "",
        "file_path": "",
    }
]


def ts() -> str:
    """Return a human-readable timestamp for logging."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

## Initialize Phoenix

In [2]:
if ENABLE_PHOENIX:

    PHOENIX_PROJECT_NAME = "raven-bot" + "-" + ts()

    try:
        from openinference.instrumentation.openai import OpenAIInstrumentor
        from phoenix.otel import register

        tracer_provider = register(
            project_name=PHOENIX_PROJECT_NAME, protocol="http/protobuf"
        )
        OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)

    except ImportError:
        print(
            "Phoenix OpenTelemetry instrumentation is not installed. please run 'uv sync'"
        )

OpenTelemetry Tracing Details
|  Phoenix Project: raven-bot-2025-11-28 16:17:05
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



## Hello LLM

Run Below cell to make sure your OPENAI_API_KEY is working

In [3]:
no_of_tips = 5
max_characters_per_tip = 140

if not os.environ.get("OPENAI_API_KEY"):
    print(f"⚠️ OPENAI_API_KEY was not found in the environment.")

# Single async OpenAI client
client = AsyncOpenAI()

MODEL_NAME = "gpt-5-nano"
REASONING_EFFORT = "minimal"  # "minimal""low" / "medium" / "high"
VERBOSITY = "low"  # keep answers short/direct

try:
    user_prompt = (
        f"I'm playing a game of Mafia (Raven) in a 24 Hour AI hackathon."
        f"Give me {no_of_tips} tips to help me win the hackathon."
        f"Keep each tip less than {max_characters_per_tip} characters."
    )
    # user_prompt = "Say Hi"
    resp = await client.responses.parse(
        model=MODEL_NAME,
        input=[
            {
                "role": "user",
                "content": user_prompt,
            }
        ],
        reasoning={"effort": REASONING_EFFORT},
        text={"verbosity": VERBOSITY},
    )
    print(resp.output_text)

except Exception as e:
    print(f"Error during OpenAI API call: {e}")
    raise

1) Listen for contradictions; trust big-picture consistency over flashy reads.  
2) Log actions and roles early; build a timeline for rapid, credible conclusions.  
3) Coordinate with teammates; share verified info, avoid reckless risk.  
4) Validate alibis with data/env clues; don’t chase weak leads.  
5) Stay adaptable; pivot strategy as new evidence emerges.


### Open AI client

In [4]:
from typing import Optional

import os

from openai import AsyncOpenAI
from pydantic import BaseModel

AI_MODEL = "gpt-5-mini"  # specify the AI model to use, switch it up and have fun
USE_STRUCTURED_OUTPUT = True  # toggle between structured vs plain LLM guesses


class GuessWord(BaseModel):
    guess: str


_ai_client: Optional[AsyncOpenAI] = None


async def _get_client() -> Optional[AsyncOpenAI]:
    """Lazy-load the OpenAI client so the notebook only instantiates it when needed."""
    global _ai_client
    if _ai_client is None:
        api_key = os.getenv("OPENAI_API_KEY")
        if not api_key:
            print(
                f"[{ts()}] ⚠️ OPENAI_API_KEY is not set. Skipping AI client initialization."
            )
            return None
        _ai_client = AsyncOpenAI(api_key=api_key)
    return _ai_client

# Helper functions

## Message model – `ParsedMessage`

In [5]:
@dataclass
class ParsedMessage:
    raw: str  # original JSON string
    type: Optional[str] = None  # message type

    # Core identifiers
    match_id: Optional[str] = None
    game_id: Optional[str] = None
    your_id: Optional[str] = None

    # Game and phase context
    day: Optional[int] = None
    phase: Optional[str] = None  # "morning" or "night"
    timeout: Optional[float] = None  # generic timeout (seconds)
    first_vote_timeout: Optional[float] = None
    time_remaining: Optional[float] = None  # used in *_player-comment, raven-comment

    # GAME START info
    your_role: Optional[str] = None
    raven_count: Optional[int] = None
    detective_count: Optional[int] = None
    doctor_count: Optional[int] = None
    villager_count: Optional[int] = None

    # Common interaction fields
    otp: Optional[str] = None
    comment: Optional[str] = None

    # Player lists / status
    all_players: List[Dict[str, Any]] = field(default_factory=list)
    villagers_alive: List[str] = field(default_factory=list)
    players_alive: List[str] = field(default_factory=list)
    discussions: List[Dict[str, Any]] = field(default_factory=list)

    # Voting information
    votes: List[str] = field(default_factory=list)  # unified votes field
    done_voting: Optional[bool] = None
    player_lynched: Optional[str] = None

    # Role-identification info (Detective / shared)
    identified_raven: List[str] = field(default_factory=list)
    identified_villager: List[str] = field(default_factory=list)

    # Comment metadata
    player_id: Optional[str] = None
    llm_model_used: Optional[str] = None

    # Ack / investigation results
    request_status: Optional[str] = None  # for generic ack
    investigated: List[str] = field(default_factory=list)  # for ack-night-investigation
    is_raven: List[bool] = field(default_factory=list)  # parallel to investigated

    # Game result
    result: Optional[str] = None

In [6]:
# In-memory game history: game_id -> list of ParsedMessage
GAME_HISTORY: Dict[str, List[ParsedMessage]] = defaultdict(list)

## 🔧 Function:  `parse_message` to parse incoming messages

In [7]:
def parse_message(msg: str) -> Optional[ParsedMessage]:
    """Parse a raw JSON string from the server into a ParsedMessage object.

    This is aligned with the latest singham protocol.md and the observed
    JSONL logs. Legacy fields like a top-level ``vote`` on incoming
    messages are no longer supported here.
    """
    try:
        data = json.loads(msg)
    except Exception as e:
        print("❌ Invalid JSON:", msg)
        print("Error:", e)
        return None

    p = ParsedMessage(raw=msg)

    # Common fields present on most messages
    p.type = data.get("type")
    p.match_id = data.get("matchId")
    p.game_id = data.get("gameId")
    p.your_id = data.get("yourId")
    p.otp = data.get("otp")
    p.day = data.get("day")
    p.phase = data.get("phase")

    # Timers
    if "timeout" in data:
        p.timeout = data.get("timeout")
    if "firstVoteTimeout" in data:
        p.first_vote_timeout = data.get("firstVoteTimeout")
    if "timeRemaining" in data:
        p.time_remaining = data.get("timeRemaining")

    # 1) GAME START
    if p.type == "game-start":
        p.raven_count = data.get("ravenCount")
        p.detective_count = data.get("detectiveCount")
        p.doctor_count = data.get("doctorCount")
        p.villager_count = data.get("villagerCount")
        p.your_role = data.get("yourRole")
        return p

    # 2) PLAYER STATUS
    if p.type == "player-status":
        # allPlayers: [{ id, isAlive?, lynchedBy, lynchedDay }, ...]
        p.all_players = data.get("allPlayers", [])
        return p

    # 3) PHASE RESULT
    if p.type == "phase-result":
        # playerLynched may be empty (no one lynched)
        lynched = data.get("playerLynched")
        p.player_lynched = lynched or None
        return p

    # 4) GENERIC ACK
    if p.type == "ack":
        p.request_status = data.get("requestStatus")
        return p

    # 5) NIGHT DISCUSSION (Ravens)
    if p.type == "night-discussion":
        p.villagers_alive = data.get("villagersAlive", [])
        return p

    # 6) RAVEN COMMENT (Server → Ravens)
    if p.type == "raven-comment":
        p.discussions = data.get("discussions", [])
        if p.discussions:
            last = p.discussions[-1]
            p.player_id = last.get("playerId")
            p.comment = last.get("comment")
            p.votes = list(last.get("votes", []))
        return p

    # 7) MORNING DISCUSSION (Villagers)
    if p.type == "morning-discussion":
        p.players_alive = data.get("playersAlive", [])
        return p

    # 8) MORNING PLAYER COMMENT (Server → Villagers)
    if p.type == "morning-player-comment":
        p.discussions = data.get("discussions", [])
        if p.discussions:
            last = p.discussions[-1]
            p.player_id = last.get("playerId")
            p.comment = last.get("comment")
            p.votes = list(last.get("votes", []))
        return p

    # 9) NIGHT INVESTIGATION (Detective)
    if p.type == "night-investigation":
        p.players_alive = data.get("playersAlive", [])
        p.identified_raven = data.get("identifiedRavens", []) or []
        p.identified_villager = data.get("identifiedVillagers", []) or []
        return p

    # 10) NIGHT PROTECTION (Doctor)
    if p.type == "night-protection":
        p.players_alive = data.get("playersAlive", [])
        return p

    # 11) ACK for detective investigation
    if p.type == "ack-night-investigation":
        p.investigated = data.get("investigated", []) or []
        p.is_raven = data.get("isRaven", []) or []
        return p

    # 12) GAME RESULT (if/when used)
    if p.type == "game-result":
        p.result = data.get("result")  # "won", "lost", "draw", etc.
        return p

    # Fallback: unknown type – still return ParsedMessage for debugging
    return p

## 🔧 Functions: `parse_outgoing_message`

In [8]:
def parse_outgoing_message(raw: str) -> ParsedMessage:
    """Parse the bot → server JSON into a ParsedMessage for logging.

    Outgoing messages now always use the ``votes`` list (no legacy ``vote`` key).
    """
    try:
        data = json.loads(raw)
    except Exception:
        # log as raw, but still return ParsedMessage
        return ParsedMessage(raw=raw)

    p = ParsedMessage(raw=raw)
    p.type = data.get("type")
    p.game_id = data.get("gameId")
    p.your_id = data.get("yourId")
    p.otp = data.get("otp")
    p.comment = data.get("comment")

    # Voting-related fields (used by all roles)
    p.votes = data.get("votes", []) or []
    p.done_voting = data.get("doneVoting")
    p.llm_model_used = data.get("llmModelUsed")

    return p

## 🔧 Functions: `print_parsed_message`

In [9]:
def print_parsed_message(p: Optional[ParsedMessage]) -> None:
    """Pretty-print the key fields of a ParsedMessage for debugging."""
    if p is None:
        print("❌ Nothing to print (parsed is None)")
        return

    print("\n================= 🧩 PARSED MESSAGE =================")
    print(f"Type           : {p.type}")
    print(f"Match ID       : {p.match_id}")
    print(f"Game ID        : {p.game_id}")
    print(f"Your ID        : {p.your_id}")
    if p.day is not None:
        print(f"Day            : {p.day}")
    if p.phase is not None:
        print(f"Phase          : {p.phase}")
    if p.timeout is not None:
        print(f"Timeout (s)    : {p.timeout}")
    if p.first_vote_timeout is not None:
        print(f"First Vote TO  : {p.first_vote_timeout}")
    if p.time_remaining is not None:
        print(f"Time Remaining : {p.time_remaining}")
    if p.otp:
        print(f"OTP            : {p.otp}")

    # GAME START
    if p.type == "game-start":
        print(f"Your Role      : {p.your_role}")
        print(f"Ravens         : {p.raven_count}")
        print(f"Detectives     : {p.detective_count}")
        print(f"Doctors        : {p.doctor_count}")
        print(f"Villagers      : {p.villager_count}")

    # PLAYER STATUS
    elif p.type == "player-status":
        print("All Players    :")
        for pl in p.all_players:
            print(f"  - {pl}")

    # PHASE RESULT
    elif p.type == "phase-result":
        print(f"Player Lynched : {p.player_lynched}")

    # ACK
    elif p.type == "ack":
        print(f"Request Status : {p.request_status}")

    # NIGHT DISCUSSION
    elif p.type == "night-discussion":
        print(f"Villagers Alive: {p.villagers_alive}")

    # RAVEN COMMENT
    elif p.type == "raven-comment":
        print("Discussions    :")
        for d in p.discussions:
            print(
                f"  - {d.get('playerId')}: {d.get('comment')} (votes={d.get('votes')})"
            )
        if p.votes:
            print(f"Last Votes     : {p.votes}")

    # MORNING DISCUSSION
    elif p.type == "morning-discussion":
        print(f"Players Alive  : {p.players_alive}")

    # MORNING PLAYER COMMENT
    elif p.type == "morning-player-comment":
        print("Discussions    :")
        for d in p.discussions:
            print(
                f"  - {d.get('playerId')}: {d.get('comment')} (votes={d.get('votes')})"
            )
        if p.votes:
            print(f"Last Votes     : {p.votes}")

    # NIGHT INVESTIGATION
    elif p.type == "night-investigation":
        print(f"Players Alive  : {p.players_alive}")
        print(f"Identified Ravens   : {p.identified_raven}")
        print(f"Identified Villagers: {p.identified_villager}")

    # NIGHT PROTECTION
    elif p.type == "night-protection":
        print(f"Players Alive  : {p.players_alive}")

    # ACK NIGHT INVESTIGATION
    elif p.type == "ack-night-investigation":
        print(f"Investigated   : {p.investigated}")
        print(f"Is Raven       : {p.is_raven}")

    # GAME RESULT
    elif p.type == "game-result":
        print(f"Game Result    : {p.result}")

    else:
        print("⚠️ No specific printer for this type; showing raw dict:")
        try:
            print(json.loads(p.raw))
        except Exception:
            print(p.raw)

    print("=====================================================")

# Role-specific responders (Villager, Raven, Detective, Doctor)

###  All Roles: `build_vote_from_morning_discussion`

In [10]:
MORNING_MSG_COUNT: Dict[str, int] = defaultdict(int)


async def build_vote_from_morning_discussion(
    parsed: ParsedMessage, done_voting: bool
) -> Dict[str, Any]:
    """
    Villager: build a vote from a morning-discussion message.
    Now uses OpenAI to choose a target + comment, with a simple random fallback.
    """
    game_id = parsed.game_id
    your_id = parsed.your_id
    otp = parsed.otp

    alive = parsed.players_alive or []
    possible_targets = [p for p in alive if p != your_id]

    vote_target = random.choice(possible_targets) if possible_targets else None
    comment = f"I think {vote_target} may be suspicious. Casting my vote."

    votes = [vote_target] if vote_target is not None else []

    msg: Dict[str, Any] = {
        "gameId": game_id,
        "yourId": your_id,
        "type": "vote",
        "otp": otp,
        "comment": comment,
        "votes": votes,
    }

    if done_voting:
        msg["doneVoting"] = True

    return msg

### Raven: `build_raven_vote_from_night_discussion`

In [11]:
from llama_index.core.prompts import PromptTemplate
from pydantic import field_validator
from typing import Union

RAVEN_SYSTEM_PROMPT = """
You are a Mafia member (Raven) in a Mafia game. It is night. You and your fellow Ravens must select exactly one ALIVE player to eliminate. Choose strategically based on threat level, influence, and behavior during the day. Your output must be a JSON object with two keys: 'kill_target' — the exact active players order in high suspicion on top. It's a list of strings, and 'comment' — a short reason for your choice (max 140 characters). Do NOT reveal Mafia identities or break secrecy.
"""

RAVEN_NIGHT_USER_PROMPT = """
Current day count: {day_count}

Alive players: 
{alive_players}

Your player name: {raven_name}

Your day time comments:
{raven_day_comments}

You will also be given the previous discussion comments on each player's day time comments.
Previous day comments:
<other_players_comments>
{other_players_comments}
</other_players_comments>

Instructions for choosing a player to eliminate:
1. Do not target your fellow Ravens.
2. Analyze the day's discussions to identify players who are close to discovering the Ravens. These are high-priority targets.
3. Consider eliminating players who are influential or leading discussions, as they can sway the villagers' votes.
4. On Day 1, if there is not much information, you can choose a player who has been quiet or acting suspiciously.
5. You need to include all alive players in your suspicion ranking, even if you don't intend to target them immediately.
6. You can ignore yourself while ranking the players based on suspicion.

You need to respond in a JSON format as follows:
{{
"kill_target": ["<player_1>", "<player_2>", "<player_3>", "..."],
"comment": "comment within 140 characters"
}}

Important:
- kill_target is a list of player IDs (strings) ranked by suspicion level (highest suspicion first)
- Include ALL alive players except yourself and fellow Ravens in the kill_target list
- The first player in the list will be your elimination choice
"""

RAVEN_NIGHT_PROMPT_TEMPLATE = PromptTemplate(RAVEN_NIGHT_USER_PROMPT)


class RavenKillChoice(BaseModel):
    kill_target: Union[str, list[str]]
    comment: str

    @field_validator("kill_target", mode="before")
    @classmethod
    def normalize_kill_target(cls, v):
        """Normalize kill_target to always be a list of strings."""
        if isinstance(v, str):
            # Split comma-separated string
            return [p.strip() for p in v.split(",") if p.strip()]
        elif isinstance(v, list):
            # Flatten any nested comma-separated strings
            result = []
            for item in v:
                if isinstance(item, str):
                    if "," in item:
                        result.extend([p.strip() for p in item.split(",") if p.strip()])
                    else:
                        if item.strip():
                            result.append(item.strip())
            return result
        return v

In [12]:
async def build_raven_vote_from_night_discussion(
    parsed: ParsedMessage,
) -> Dict[str, Any]:
    """
    Raven: build a vote message from a night-discussion message.
    Uses LLM to pick villagers to target (votes list).
    """
    game_id = parsed.game_id
    your_id = parsed.your_id
    otp = parsed.otp

    villagers = sorted(parsed.villagers_alive or [])

    client = await _get_client()
    if not client:
        # Fallback to all villagers if AI client is not available
        votes = villagers[:]
        comment = "As Raven, I will vote for all available villagers."
        return {
            "gameId": game_id,
            "yourId": your_id,
            "type": "vote",
            "otp": otp,
            "comment": comment,
            "votes": votes,
            "llmModelUsed": AI_MODEL,
        }

    day_count = parsed.day or 1
    raven_name = your_id or "Raven"
    alive_players = "\n".join(villagers)

    # Get all day-time discussions from history
    all_day_discussions = []
    if game_id in GAME_HISTORY:
        for past_message in GAME_HISTORY[game_id]:
            if (
                past_message.type == "morning-player-comment"
                and past_message.discussions
            ):
                all_day_discussions.extend(past_message.discussions)

    raven_day_comments = "\n".join(
        [
            f'{d.get("playerId")}: {d.get("comment")}'
            for d in all_day_discussions
            if d.get("playerId") == your_id
        ]
    )
    other_players_comments = "\n".join(
        [
            f'{d.get("playerId")}: {d.get("comment")}'
            for d in all_day_discussions
            if d.get("playerId") != your_id
        ]
    )

    raven_system_prompt = RAVEN_SYSTEM_PROMPT
    raven_user_prompt = RAVEN_NIGHT_PROMPT_TEMPLATE.format(
        day_count=day_count,
        alive_players=alive_players,
        raven_name=raven_name,
        raven_day_comments=raven_day_comments,
        other_players_comments=other_players_comments,
    )

    try:
        response = await client.responses.parse(
            model="gpt-5-mini",
            input=[
                {"role": "system", "content": raven_system_prompt},
                {"role": "user", "content": raven_user_prompt},
            ],
            reasoning={"effort": "low"},
            text={"verbosity": "low"},
            text_format=RavenKillChoice,
        )
        raven_response_payload = response.output_parsed
    except Exception as exc:
        print(f"[{ts()}] ⚠️ Raven AI guess failed: {exc}")
        # Fallback to all villagers on failure
        votes = villagers[:]
        comment = "As Raven, I will vote for all available villagers (AI fallback)."
        return {
            "gameId": game_id,
            "yourId": your_id,
            "type": "vote",
            "otp": otp,
            "comment": comment,
            "votes": votes,
            "llmModelUsed": AI_MODEL,
        }

    if not raven_response_payload:
        print(f"[{ts()}] ⚠️ Raven AI response was empty.")
        votes = villagers[:]
        comment = "As Raven, I will vote for all available villagers (AI empty)."
    else:
        # Parse kill_target - handle both proper list and comma-separated string
        kill_target_raw = raven_response_payload.kill_target
        print(
            f"[{ts()}] 🔍 DEBUG: kill_target_raw type={type(kill_target_raw)}, value={kill_target_raw}"
        )

        votes = []

        # Process the kill_target to ensure it's a proper list
        if isinstance(kill_target_raw, list):
            # Flatten any nested structure and split comma-separated strings
            for item in kill_target_raw:
                if isinstance(item, str):
                    # Check if this string contains commas
                    if "," in item:
                        # Split by comma and add each part
                        votes.extend([p.strip() for p in item.split(",") if p.strip()])
                    else:
                        # Single player ID
                        if item.strip():
                            votes.append(item.strip())
        elif isinstance(kill_target_raw, str):
            # Single string - split by commas
            votes = [p.strip() for p in kill_target_raw.split(",") if p.strip()]
        else:
            # Unexpected format
            print(f"[{ts()}] ⚠️ Unexpected kill_target format: {kill_target_raw}")
            votes = villagers[:] if villagers else []

        print(f"[{ts()}] 🔍 DEBUG: Final votes after parsing={votes}")
        comment = raven_response_payload.comment.strip()

    # If LLM gave no votes but there are villagers, fall back to all
    if not votes and villagers:
        votes = villagers[:]
        comment = (
            comment + " (fallback: voting for all available villagers)"
            if comment
            else "As Raven, I will vote for all available villagers."
        )

    return {
        "gameId": game_id,
        "yourId": your_id,
        "type": "vote",
        "otp": otp,
        "comment": comment,
        "votes": votes,
        "llmModelUsed": AI_MODEL,
    }

### Detective: `build_detective_vote_from_night_investigation`

In [13]:
from llama_index.core.prompts import PromptTemplate


DETECTIVE_SYSTEM_PROMPT = """
You are the Detective in a Mafia game. It is night. You may investigate exactly one ALIVE player to learn if they are Mafia or not. Your output must be a JSON object with two keys: 'target_player' — the exact name of one alive player you choose to investigate, and 'comment' — a short reason for your choice (max 140 characters). Do NOT reveal any hidden roles directly. Only decide who to investigate based on the user's provided information and reasoning.
"""


DETECTIVE_NIGHT_USER_PROMPT = """

Current day count: {day_count}

Alive players: 
{alive_players}

Your player name: {detective_name}

Your day time comments:
{detective_day_comments}

previous night investigations:
{previous_night_investigations}


You will also given with the previous discussion comments on each player's day time comments
Previous day comments:
<other_players_comments>
{other_players_comments}
</other_players_comments>

Instructions to choose the player to investigate:
1. You shouldn't investigate yourself.
2. On Day 1, choose a player randomly from the alive players excluding yourself.
3. Consider players' day time comments to identify suspicious behavior.
4. Choose one player to investigate based on the above information.
5. Don't investigate the same player again, you can refer the previous night investigations to identify who you have already investigated.
6. You can use the `other_players_comments` section to understand the behavior of other players during the day and if you feel any player is suspicious based on their comments, you can choose to investigate them.

You need to respond in a json format as follows:
{
"target_player": "<name_of_player>",
"comment": "comment within 140 characters"
}
"""

DETECTIVE_NIGHT_USER_PROMPT_TEMPLATE = PromptTemplate(DETECTIVE_NIGHT_USER_PROMPT)

In [14]:
class GuessTargetPlayer(BaseModel):
    target_player: str
    comment: str

In [15]:
async def build_detective_vote_from_night_investigation(
    parsed: ParsedMessage,
) -> Dict[str, Any]:
    """Detective: build a vote from a night-investigation message."""
    game_id = parsed.game_id
    your_id = parsed.your_id
    otp = parsed.otp

    day_count = parsed.day or 1
    detective_name = your_id or "Detective"
    alive_players = "\n".join(parsed.players_alive or [])
    # previous_night_investigations = parsed.discussions or []
    previous_night_villagers = parsed.identified_villager
    previous_night_ravens = parsed.identified_raven

    previous_night_investigations = [
        {
            "identified_villagers": "\n".join(previous_night_villagers),
            "identified_ravens": "\n".join(previous_night_ravens),
        }
    ]

    # Get all day-time discussions from history
    all_day_discussions = []
    if game_id in GAME_HISTORY:
        for past_message in GAME_HISTORY[game_id]:
            if (
                past_message.type == "morning-player-comment"
                and past_message.discussions
            ):
                all_day_discussions.extend(past_message.discussions)

    detective_day_comments = "\n".join(
        [
            f'{d.get("playerId")}: {d.get("comment")}'
            for d in all_day_discussions
            if d.get("playerId") == your_id
        ]
    )
    other_players_comments = "\n".join(
        [
            f'{d.get("playerId")}: {d.get("comment")}'
            for d in all_day_discussions
            if d.get("playerId") != your_id
        ]
    )

    print("Previous night investigations:", previous_night_investigations)

    alive = parsed.players_alive or []
    safe_players = set(parsed.identified_villager or [])

    possible_targets = [p for p in alive if p not in safe_players]
    if not possible_targets:
        possible_targets = alive[:]

    # vote_target = random.choice(possible_targets) if possible_targets else None
    # comment = (
    #     f"As Detective, I want to investigate {vote_target}. Casting my vote on them."
    # )
    detective_system_prompt = DETECTIVE_SYSTEM_PROMPT
    # other_players_comments = "\n".join(
    #     [
    #         f"{d.get('playerId')}: {d.get('comment', '')}"
    #         for d in parsed.discussions
    #         if d.get("playerId") != your_id
    #     ]
    # )
    detective_user_prompt = DETECTIVE_NIGHT_USER_PROMPT_TEMPLATE.format(
        day_count=day_count,
        alive_players="\n".join(alive),
        detective_name=detective_name,
        detective_day_comments=detective_day_comments,
        previous_night_investigations=json.dumps(previous_night_investigations),
        other_players_comments=other_players_comments,
    )

    detective_response = await client.responses.parse(
        model=AI_MODEL,
        input=[
            {
                "role": "system",
                "content": detective_system_prompt,
            },
            {
                "role": "user",
                "content": detective_user_prompt,
            },
        ],
        reasoning={"effort": "low"},
        text={"verbosity": "low"},
        text_format=GuessTargetPlayer,
    )

    detective_response_payload = getattr(detective_response, "output_parsed", None)

    vote_target = detective_response_payload.target_player.strip()
    comment = detective_response_payload.comment.strip()

    votes = [vote_target] if vote_target is not None else []

    return {
        "gameId": game_id,
        "yourId": your_id,
        "type": "vote",
        "otp": otp,
        "comment": comment,
        "votes": votes,
    }

### Doctor: `build_doctor_vote_from_night_protection`

In [16]:
from llama_index.core.prompts import PromptTemplate

DOCTOR_SYSTEM_PROMPT = """
You are the Doctor in a Mafia game. It is night. You can choose exactly one ALIVE player to protect from the Mafia kill. 
Your output must be a JSON object with two keys: 
- 'protect_target' — the exact name of one alive player, and 
- 'comment' — a short reason (max 140 characters). Do NOT reveal hidden roles or predict future actions. Only decide who to protect based on the user's rationale.
"""


DOCTOR_NIGHT_USER_PROMPT = """

Current day count: {day_count}

Alive players: 
{alive_players}

Your player name: {doctor_name}

Your day time comments:
{doctor_day_comments}


Provide your protection choice. Think based on suspicion levels, behavior during day discussion, and who is most valuable to keep alive. Output only JSON with 'protect_target' and 'comment'.

You will also given with the previous discussion comments on each player's day time comments
Previous day comments:
<other_players_comments>
{other_players_comments}
</other_players_comments>


Important instructions:
1. During Day 1 and Day 2, prioritize protecting yourself because there might be chance of being targeted early by the Ravens. So first two days, protect yourself and play safely.
2. From the third day onwards, consider protecting other players based on their behavior and the overall game dynamics.
3. If you have identified any Ravens, avoid protecting them.
4. Consider protecting: players who have been very vocal or have drawn attention during the day discussions, as they might be targeted by Ravens.
5. If you have provided any valuable insights during the day, consider protecting yourself to ensure you can continue contributing, because Ravens may target you and try to eliminate influential players like you. So protect yourself if you have been playing a key role in discussions, so you can keep influencing the game.

You need to respond in a json format as follows:
{
"protect_target": "<name_of_player>",
"comment": "comment within 140 characters"
}
"""

DOCTOR_NIGHT_PROMPT_TEMPLATE = PromptTemplate(DOCTOR_NIGHT_USER_PROMPT)


DOCTOR_DAY_USER_PROMPT = """

You are a Doctor in a game of Mafia (Raven). During the day phase, your role is to participate in discussions and help identify Ravens among the villagers.

Analyze the ongoing discussions and provide your insights to assist the villagers in making informed decisions about whom to trust and whom to suspect.

Just return the comment within 140 characters.
You must return your response within the character limit.

"""

DOCTOR_DAY_PROMPT_TEMPLATE = PromptTemplate(DOCTOR_DAY_USER_PROMPT)

In [17]:
class DoctorGuessProtection(BaseModel):
    protect_target: str
    comment: str

In [18]:
async def build_doctor_vote_from_night_protection(
    parsed: ParsedMessage,
) -> Dict[str, Any]:
    """Doctor: build a protection vote from a night-protection message."""
    game_id = parsed.game_id
    your_id = parsed.your_id
    otp = parsed.otp

    alive = parsed.players_alive or []

    client = await _get_client()
    # if not client:
    #     # Fallback to random choice if AI client is not available
    #     protect_target = random.choice(alive) if alive else None
    #     comment = f"As Doctor, I choose to protect {protect_target} tonight."
    #     votes = [protect_target] if protect_target is not None else []
    #     return {
    #         "gameId": game_id,
    #         "yourId": your_id,
    #         "type": "vote",
    #         "otp": otp,
    #         "comment": comment,
    #         "votes": votes,
    #     }

    doctor_name = your_id or "Doctor"
    day_count = parsed.day or 1
    alive_players = "\n".join(parsed.players_alive or [])

    # Get all day-time discussions from history
    all_day_discussions = []
    if game_id in GAME_HISTORY:
        for past_message in GAME_HISTORY[game_id]:
            if (
                past_message.type == "morning-player-comment"
                and past_message.discussions
            ):
                all_day_discussions.extend(past_message.discussions)

    doctor_day_comments = "\n".join(
        [
            f'{d.get("playerId")}: {d.get("comment")}'
            for d in all_day_discussions
            if d.get("playerId") == your_id
        ]
    )
    other_players_comments = "\n".join(
        [
            f'{d.get("playerId")}: {d.get("comment")}'
            for d in all_day_discussions
            if d.get("playerId") != your_id
        ]
    )

    doctor_system_prompt = DOCTOR_SYSTEM_PROMPT
    doctor_night_user_prompt = DOCTOR_NIGHT_PROMPT_TEMPLATE.format(
        day_count=day_count,
        alive_players=alive_players,
        doctor_name=doctor_name,
        doctor_day_comments=doctor_day_comments,
        other_players_comments=other_players_comments,
    )

    # try:
    response = await client.responses.parse(
        model=AI_MODEL,
        input=[
            {
                "role": "system",
                "content": doctor_system_prompt,
            },
            {
                "role": "user",
                "content": doctor_night_user_prompt,
            },
        ],
        reasoning={"effort": "low"},
        text={"verbosity": "low"},
        text_format=DoctorGuessProtection,
    )
    doctor_response_payload = response.output_parsed
    # except Exception as exc:
    #     print(f"[{ts()}] ⚠️ AI guess failed: {exc}")
    #     # Fallback to random choice on failure
    #     protect_target = random.choice(alive) if alive else None
    #     comment = (
    #         f"As Doctor, I choose to protect {protect_target} tonight (AI fallback)."
    #     )
    #     votes = [protect_target] if protect_target is not None else []
    #     return {
    #         "gameId": game_id,
    #         "yourId": your_id,
    #         "type": "vote",
    #         "otp": otp,
    #         "comment": comment,
    #         "votes": votes,
    #     }

    # if not doctor_response_payload:
    #     print(f"[{ts()}] ⚠️ AI response was empty.")
    #     protect_target = random.choice(alive) if alive else None
    #     comment = f"As Doctor, I choose to protect {protect_target} tonight (AI empty)."
    # else:
    protect_target = doctor_response_payload.protect_target.strip()
    comment = doctor_response_payload.comment.strip()

    votes = [protect_target] if protect_target is not None else []

    return {
        "gameId": game_id,
        "yourId": your_id,
        "type": "vote",
        "otp": otp,
        "comment": comment,
        "votes": votes,
    }

## Logging – in-memory + JSONL file

In [19]:
# In-memory game logs: game_id -> list of events
GAME_LOGS: Dict[str, List[Dict[str, Any]]] = defaultdict(list)

# In-memory game history: game_id -> list of ParsedMessage
GAME_HISTORY: Dict[str, List[ParsedMessage]] = defaultdict(list)


# Global event counter (across the whole run)
EVENT_COUNTER = itertools.count(1)

# Log file setup
LOG_DIR = "logs"
os.makedirs(LOG_DIR, exist_ok=True)

LOG_FILE_PATH = os.path.join(
    LOG_DIR, f"raven_events_{datetime.now().strftime('%Y%m%d_%H%M%S')}.jsonl"
)

print(f"Logging events to: {LOG_FILE_PATH}")


def record_event(
    *, direction: str, raw: str, parsed: Optional[ParsedMessage] = None
) -> None:
    """Record a single event both in memory and in the JSONL log file."""
    seq = next(EVENT_COUNTER)
    timestamp = ts()

    game_id = getattr(parsed, "game_id", None) if parsed is not None else None
    match_id = getattr(parsed, "match_id", None) if parsed is not None else None
    msg_type = getattr(parsed, "type", None) if parsed is not None else None

    if not game_id:
        game_id = "__no_game_id__"

    event = {
        "seq": seq,
        "ts": timestamp,
        "direction": direction,  # "IN" or "OUT"
        "type": msg_type,
        "matchId": match_id,
        "gameId": game_id,
        "raw": raw,
    }

    # 1) In-memory
    GAME_LOGS[game_id].append(event)

    # 2) On disk (append JSONL)
    with open(LOG_FILE_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(event) + "\n")


def record_parsed_message(parsed: ParsedMessage):
    """Record a parsed message to the in-memory history."""
    if parsed and parsed.game_id:
        GAME_HISTORY[parsed.game_id].append(parsed)


def print_game_log_from_memory(game_id: str) -> None:
    """Print all events for a gameId from in-memory logs."""
    events = GAME_LOGS.get(game_id, [])
    if not events:
        print(f"No events logged for gameId={game_id}")
        return

    print("\n" + "=" * 80)
    print(f"🎮 GAME LOG (in-memory) for gameId={game_id}")
    print("=" * 80)

    for ev in events:
        direction = "Server → Bot" if ev["direction"] == "IN" else "Bot → Server"
        print(
            f"\n#{ev['seq']} [{ev['ts']}] {direction} "
            f"(type={ev['type']}, matchId={ev['matchId']})"
        )
        print("-" * 80)
        try:
            obj = json.loads(ev["raw"])
            print(json.dumps(obj, indent=2))
        except Exception:
            print(ev["raw"])

    print("\n" + "=" * 80 + "\n")

Logging events to: logs\raven_events_20251128_161721.jsonl


# Main async loop – connect, parse, respond, log

### How the non-blocking LLM logic works

This section is the **heart of the bot's concurrency model**. The goal is:

> Keep listening to the server all the time, without waiting for the LLM.

We do this in two pieces:

1. **`connect_parse_respond_forever` (main loop)**  
   - Opens the WebSocket connection to the game server.  
   - Repeatedly does `ws.recv()` to get the next message.  
   - Parses and logs the message.  
   - For messages that need an LLM decision, it calls:

     ```python
     asyncio.create_task(handle_message_and_respond(parsed, ws, send_lock))
     ```

   - This line *starts* an async task and immediately returns, so the loop
     can go back to waiting for the next message.

2. **`handle_message_and_respond` (background task)**  
   - Runs **in the background**, one task per message that needs the LLM.  
   - Decides what to do based on `parsed.type` (Villager / Raven / Detective / Doctor).  
   - Calls the appropriate async helper which internally talks to OpenAI.  
   - Builds the outgoing JSON payload, logs it, and sends it via `ws.send(...)`.  
   - Uses a shared `send_lock` so that only one task writes to the WebSocket at a time.

Because of this design:

- The bot can receive and handle **multiple games or phases in parallel**.
- A slow LLM call does **not** block the WebSocket; other messages keep flowing.
- The pattern is simple and beginner-friendly: one main loop + one background handler.


## Handler `handle_message_and_respond`

In [20]:
async def handle_message_and_respond(
    parsed: ParsedMessage, ws, send_lock: asyncio.Lock
) -> None:
    """
    Handle a single parsed message **in a background task**.

    This function is where we decide whether we need to call the LLM and send
    a response back to the server. It is meant to be launched with
    `asyncio.create_task(...)`, so the main WebSocket receive loop does not
    wait for the LLM call to finish.

    Parameters
    ----------
    parsed : ParsedMessage
        The already-parsed server message.
    ws : websockets.WebSocketClientProtocol
        The live WebSocket connection back to the game server.
    send_lock : asyncio.Lock
        A lock so that only one task calls `ws.send(...)` at a time.
    """
    try:
        if parsed is None:
            return

        outgoing: Optional[Dict[str, Any]] = None

        # --- Decide what to do based on message type ---
        if parsed.type == "morning-discussion":
            # All Players: build a vote for morning discussion
            outgoing = await build_vote_from_morning_discussion(
                parsed, done_voting=True
            )

        elif parsed.type == "night-discussion":
            # Raven: build votes for night elimination
            outgoing = await build_raven_vote_from_night_discussion(parsed)

        elif parsed.type == "night-investigation":
            # Detective: choose a target to investigate
            outgoing = await build_detective_vote_from_night_investigation(parsed)

        elif parsed.type == "night-protection":
            # Doctor: choose someone to protect
            outgoing = await build_doctor_vote_from_night_protection(parsed)

        else:
            # For all other message types (acks, results, etc.) we just log.
            print(f"[{ts()}] ℹ️ No action needed/handled for type={parsed.type!r}")
            return

        if not outgoing:
            # Nothing to send (e.g., no valid targets)
            print(f"[{ts()}] ℹ️ No outgoing message built for type={parsed.type!r}")
            return

        # --- Serialize and log OUT message ---
        raw_out = json.dumps(outgoing)
        out_parsed = parse_outgoing_message(raw_out)
        record_event(direction="OUT", raw=raw_out, parsed=out_parsed)

        # Only one task should call ws.send at a time → use a lock.
        async with send_lock:
            await ws.send(raw_out)

        print(f"[Bot → sent]\n{json.dumps(outgoing, indent=2)}")

    except Exception as e:
        # Make sure background task failures are visible
        print(f"[{ts()}] ⚠️ Error in handle_message_and_respond: {e}")

## Main Loop `connect_parse_respond_forever`

In [21]:
async def connect_parse_respond_forever():
    """
    Main loop: keep the WebSocket connection open, keep *listening* for messages,
    and spin off background tasks to handle any LLM / decision work.

    The key idea:
    - This loop ONLY waits on `ws.recv()` and other cheap operations.
    - For any message that needs an LLM call, we do:

          asyncio.create_task(handle_message_and_respond(parsed, ws, send_lock))

      so the loop can immediately go back to listening for the next message.
    """
    print(f"[{ts()}] 🔌 Connecting to {WS_URL} ...")
    try:
        async with websockets.connect(WS_URL, open_timeout=CONNECT_TIMEOUT) as ws:
            print(f"[{ts()}] ✅ Connection established.")

            # One lock shared by all background tasks that want to send on this websocket
            send_lock = asyncio.Lock()

            while True:
                try:
                    # 1) Wait for the next message from the server
                    msg = await asyncio.wait_for(ws.recv(), timeout=RECV_TIMEOUT)

                    # 2) Log and parse incoming message
                    print(f"\n[Server → raw] {msg}")
                    parsed = parse_message(msg)
                    print_parsed_message(parsed)

                    # Record the IN event (even if parsing failed, we capture raw)
                    record_event(direction="IN", raw=msg, parsed=parsed)

                    if parsed:
                        record_parsed_message(parsed)

                    if parsed is None:
                        # Invalid/unknown message, nothing more to do
                        continue

                    # 3) Fire-and-forget background task to handle LLM + response.
                    #    This is what makes the LLM call *non-blocking* for the main loop.
                    asyncio.create_task(
                        handle_message_and_respond(parsed, ws, send_lock)
                    )

                except asyncio.TimeoutError:
                    if KEEP_ALIVE:
                        # No messages recently, but keep the connection open.
                        continue
                    print(
                        f"[{ts()}] ⏹️ No messages within {RECV_TIMEOUT}s; closing connection."
                    )
                    break
                except websockets.exceptions.ConnectionClosedOK:
                    print(f"[{ts()}] 🔒 Connection closed by server (OK).")
                    break
                except websockets.exceptions.ConnectionClosedError as e:
                    print(f"[{ts()}] ❌ Connection closed with error: {e}")
                    break
                except Exception as e:
                    print(f"[{ts()}] ⚠️ Unexpected error while listening: {e}")
                    break
    except Exception as e:
        print(f"[{ts()}] ❌ Connection failed: {e}")

# Run and Debug

## Run the bot

In [ ]:
# if the bot_task is already running, run bot_task.cancel() first the old task
try:
    bot_task.cancel()
except Exception:
    pass

# Start a (new) bot task
bot_task = asyncio.create_task(connect_parse_respond_forever())

[2025-11-28 16:17:31] 🔌 Connecting to ws://localhost:2025 ...
[2025-11-28 16:17:33] ✅ Connection established.
[2025-11-28 16:17:33] ✅ Connection established.

[Server → raw] {"matchId":"5A33F569-30A6-4FBE-B1C2-07548E228204","gameId":"7DEF5D18-F89A-41F5-BD93-6926E849A127","gameNumber":"Game1","yourId":"P3","type":"game-start","ravenCount":2,"detectiveCount":1,"doctorCount":1,"villagerCount":4,"yourRole":"Raven"}

================= 🧩 PARSED MESSAGE =================
Type           : game-start
Match ID       : 5A33F569-30A6-4FBE-B1C2-07548E228204
Game ID        : 7DEF5D18-F89A-41F5-BD93-6926E849A127
Your ID        : P3
Your Role      : Raven
Ravens         : 2
Detectives     : 1
Doctors        : 1
Villagers      : 4
[2025-11-28 16:17:37] ℹ️ No action needed/handled for type='game-start'

[Server → raw] {"matchId":"5A33F569-30A6-4FBE-B1C2-07548E228204","gameId":"7DEF5D18-F89A-41F5-BD93-6926E849A127","gameNumber":"Game1","yourId":"P3","type":"game-start","ravenCount":2,"detectiveCount":1,"

In [23]:
bot_task.cancel()

True

## Print Games logs

In [24]:
# game_ids = list(GAME_LOGS.keys())
# game_ids

In [25]:
# print_game_log_from_memory(game_ids[0])

## Run the Below cell stop the task

In [26]:
# bot_task.cancel()